# 1. Business Problem

This project analyses the sales pipeline of a B2B technology company that sells several technology products to business customers.

The main objective is to understand how effectively the company is converting its sales pipeline into revenue and what factors are associated with stronger or weaker sales performance.


In [264]:
#Importing necessary libraries
import pandas as pd

df = pd.read_csv("../data/sales_pipeline.csv")
print("First 5 Rows:")

print(df.head(11))
print("\n Dataframe information:")
df.info()



First 5 Rows:
   opportunity_id      sales_agent         product     account deal_stage  \
0        1C1I7A6R      Moses Frase  GTX Plus Basic     Cancity        Won   
1        Z063OYW0  Darcel Schlecht          GTXPro       Isdom        Won   
2        EC4QE1BX  Darcel Schlecht      MG Special     Cancity        Won   
3        MV1LWRNH      Moses Frase       GTX Basic     Codehow        Won   
4        PE84CX4O        Zane Levy       GTX Basic      Hatfan        Won   
5        ZNBS69V1    Anna Snelling      MG Special    Ron-tech        Won   
6        9ME3374G   Vicki Laflamme      MG Special     J-Texon        Won   
7        7GN8Q4LL   Markita Hansen       GTX Basic      Cheers        Won   
8        OLK9LKZB  Niesha Huffines  GTX Plus Basic    Zumgoity        Won   
9        HAXMC4IX   James Ascencio     MG Advanced         NaN   Engaging   
10       NL3JZH1Z    Anna Snelling      MG Special  Bioholding        Won   

   engage_date  close_date  close_value  
0   2016-10-20  201

## 2. Dataset Overview

The main dataset contains 8,800 sales opportunities and includes information about sales agents, products, accounts, deal stages, engagement dates, closing dates and closing values.

The dataset is used to investigate how effectively the company is converting its sales pipeline into revenue and to identify factors associated with stronger or weaker sales performance.


In [334]:
#Investigating the number of rows, the columns and opportunities 
df.info()
df.columns

<class 'pandas.DataFrame'>
RangeIndex: 8800 entries, 0 to 8799
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   opportunity_id  8800 non-null   str    
 1   sales_agent     8800 non-null   str    
 2   product         8800 non-null   str    
 3   account         7375 non-null   str    
 4   deal_stage      8800 non-null   str    
 5   engage_date     8300 non-null   str    
 6   close_date      6711 non-null   str    
 7   close_value     6711 non-null   float64
dtypes: float64(1), str(7)
memory usage: 1.1 MB


Index(['opportunity_id', 'sales_agent', 'product', 'account', 'deal_stage',
       'engage_date', 'close_date', 'close_value'],
      dtype='str')

## 3. Data Quality Investigation

Before analysing sales performance, the dataset is checked for missing values and other potential data-quality issues.

The aim is to understand whether missing information represents a genuine data-quality problem or whether it is related to the current stage of an opportunity.


In [335]:
df["deal_stage"].unique()

<ArrowStringArray>
['Won', 'Engaging', 'Lost', 'Prospecting']
Length: 4, dtype: str

In [336]:
df["deal_stage"].value_counts()

deal_stage
Won            4238
Lost           2473
Engaging       1589
Prospecting     500
Name: count, dtype: int64

In [337]:
df.isnull().sum()

opportunity_id       0
sales_agent          0
product              0
account           1425
deal_stage           0
engage_date        500
close_date        2089
close_value       2089
dtype: int64

In [338]:
df[df["deal_stage"]=="Engaging"]["close_date"].isnull().sum()

np.int64(1589)

In [339]:
df[df["deal_stage"]=="Prospecting"]["close_date"].isnull().sum()

np.int64(500)

In [340]:
df[df["deal_stage"]=="Won"]["close_date"].isnull().sum()

np.int64(0)

In [341]:
df[df["deal_stage"]=="Lost"]["close_date"].isnull().sum()

np.int64(0)

In [342]:
df[df["account"].isnull()]["deal_stage"].value_counts()

deal_stage
Engaging       1088
Prospecting     337
Name: count, dtype: int64

### Missing Value Investigation

The dataset contains 2,089 missing values in both `close_date` and `close_value`. These records correspond exactly to the 1,589 Engaging and 500 Prospecting opportunities.

All Won and Lost opportunities have a recorded `close_date`.

This suggests that the missing `close_date` and `close_value` values are related to opportunities that have not yet reached a closed stage, rather than necessarily representing missing or incorrect data.


### Account Data Quality

There are 1,425 opportunities with missing account information.
All of these records are in the Engaging or Prospecting stages.

This means the missing account information is concentrated entirely
within open opportunities. The reason for this pattern should be
investigated before deciding whether the missing values require
treatment.

## 4. Pipeline Health

The first stage of the analysis is to understand the overall health of the sales pipeline.

This includes examining the distribution of opportunities across deal stages, identifying how many opportunities are currently open, and measuring the value and conversion of closed opportunities.

The aim is to understand how much of the current pipeline has been converted into Won opportunities and how much remains open or has been Lost.


In [343]:
df["close_value"].sum()

np.float64(10005534.0)

In [344]:
df[df["deal_stage"]=="Won"]["close_value"].sum()

np.float64(10005534.0)

In [345]:
df[df["deal_stage"]=="Lost"]["close_value"].sum()

np.float64(0.0)

In [346]:
df[df["deal_stage"]=="Won"]["close_value"].mean()

np.float64(2360.9093912222747)

## 5. Product Performance

Product performance is analysed to identify which products contribute most to Won opportunities and recorded Won value.

The analysis compares deal volume, total Won value, average Won deal value and win rate to understand differences between products.


In [347]:
value_count_won = df[df["deal_stage"]=="Won"]["opportunity_id"].count()
value_count_lost = df[df["deal_stage"]=="Lost"]["opportunity_id"].count()

percentage_for_value_count = value_count_won / (value_count_won + value_count_lost) * 100
print(percentage_for_value_count)

63.15005215318135


In [348]:
df[df["deal_stage"]=="Won"].groupby("product")["close_value"].sum()

product
GTK 500            400612.0
GTX Basic          499263.0
GTX Plus Basic     705275.0
GTX Plus Pro      2629651.0
GTXPro            3510578.0
MG Advanced       2216387.0
MG Special          43768.0
Name: close_value, dtype: float64

In [349]:
df[df["deal_stage"]=="Won"].groupby("product")["opportunity_id"].count()

product
GTK 500            15
GTX Basic         915
GTX Plus Basic    653
GTX Plus Pro      479
GTXPro            729
MG Advanced       654
MG Special        793
Name: opportunity_id, dtype: int64

In [350]:
df[df["deal_stage"]=="Won"].groupby("product")["close_value"].mean()

product
GTK 500           26707.466667
GTX Basic           545.642623
GTX Plus Basic     1080.053599
GTX Plus Pro       5489.876827
GTXPro             4815.607682
MG Advanced        3388.970948
MG Special           55.192938
Name: close_value, dtype: float64

In [351]:
df[(df["deal_stage"] == "Won") & (df["product"] == "GTK 500")]["close_value"]

59      25897.0
677     30288.0
1156    29617.0
1825    25288.0
2559    23746.0
2586    24949.0
2808    27385.0
3352    24745.0
3495    24899.0
3522    25791.0
3839    29166.0
3925    25464.0
6420    26186.0
7848    27971.0
8078    29220.0
Name: close_value, dtype: float64

In [352]:
df[df["product"] == "GTK 500"]["deal_stage"].value_counts()

deal_stage
Won         15
Engaging    15
Lost        10
Name: count, dtype: int64

In [353]:
Won_by_product = df[df["deal_stage"] == "Won"].groupby("product")["opportunity_id"].count()


In [354]:
lost_by_product = df[df["deal_stage"] == "Lost"].groupby("product")["opportunity_id"].count()

In [355]:
win_rate_by_product = Won_by_product/(Won_by_product+lost_by_product)*100
print(win_rate_by_product)

product
GTK 500           60.000000
GTX Basic         63.718663
GTX Plus Basic    62.131304
GTX Plus Pro      64.295302
GTXPro            63.557105
MG Advanced       60.332103
MG Special        64.840556
Name: opportunity_id, dtype: float64


In [356]:
df[df["deal_stage"] == "Won"].groupby("product")["close_value"].sum()

product
GTK 500            400612.0
GTX Basic          499263.0
GTX Plus Basic     705275.0
GTX Plus Pro      2629651.0
GTXPro            3510578.0
MG Advanced       2216387.0
MG Special          43768.0
Name: close_value, dtype: float64

In [357]:
won_value_by_product = df[df["deal_stage"] == "Won"].groupby("product")["close_value"].sum()
total_sum = df[df["deal_stage"]=="Won"]["close_value"].sum()
product_percentage_share = won_value_by_product/(total_sum)*100
print(product_percentage_share)

product
GTK 500            4.003904
GTX Basic          4.989869
GTX Plus Basic     7.048849
GTX Plus Pro      26.281966
GTXPro            35.086363
MG Advanced       22.151611
MG Special         0.437438
Name: close_value, dtype: float64


## 6. Sales Performance

Sales performance is analysed at individual sales-agent level to understand differences in deal volume, Won value, average Won deal value and win rate.

Comparing these measures helps identify whether strong performance is driven by closing a larger number of opportunities, generating higher-value deals, or a combination of both.


In [358]:
total_deal_stage_count = df["deal_stage"].value_counts()

engaging_deal = total_deal_stage_count["Engaging"]
prospecting_deal = total_deal_stage_count["Prospecting"]

deal_stage_unclosed = engaging_deal + prospecting_deal 
print(deal_stage_unclosed)

percetange_of_deal_stage_unclosed = deal_stage_unclosed/df["opportunity_id"].count()*100
print(percetange_of_deal_stage_unclosed)

2089
23.738636363636363


In [359]:
percentage_of_deals_engaged = engaging_deal/(deal_stage_unclosed)*100
print(percentage_of_deals_engaged)

76.06510292005744


In [360]:
df[df["deal_stage"]=="Won"].groupby("sales_agent")["opportunity_id"].count()

sales_agent
Anna Snelling         208
Boris Faz             101
Cassey Cress          163
Cecily Lampkin        107
Corliss Cosme         150
Daniell Hammack       114
Darcel Schlecht       349
Donn Cantrell         158
Elease Gluck           80
Garret Kinder          75
Gladys Colclough      135
Hayden Neloms         107
James Ascencio        135
Jonathan Berthelot    171
Kami Bicknell         174
Kary Hendrixson       209
Lajuana Vencill       127
Markita Hansen        130
Marty Freudenburg     122
Maureen Marcano       149
Moses Frase           129
Niesha Huffines       105
Reed Clapper          155
Rosalina Dieter        72
Rosie Papadopoulos     78
Versie Hillebrand     176
Vicki Laflamme        221
Violet Mclelland      122
Wilburn Farren         55
Zane Levy             161
Name: opportunity_id, dtype: int64

In [361]:
df[df["deal_stage"]=="Won"].groupby("sales_agent")["close_value"].sum()

sales_agent
Anna Snelling          275056.0
Boris Faz              261631.0
Cassey Cress           450489.0
Cecily Lampkin         229800.0
Corliss Cosme          421036.0
Daniell Hammack        364229.0
Darcel Schlecht       1153214.0
Donn Cantrell          445860.0
Elease Gluck           289195.0
Garret Kinder          197773.0
Gladys Colclough       345674.0
Hayden Neloms          272111.0
James Ascencio         413533.0
Jonathan Berthelot     284886.0
Kami Bicknell          316456.0
Kary Hendrixson        454298.0
Lajuana Vencill        194632.0
Markita Hansen         328792.0
Marty Freudenburg      291195.0
Maureen Marcano        350395.0
Moses Frase            207182.0
Niesha Huffines        176961.0
Reed Clapper           438336.0
Rosalina Dieter        235403.0
Rosie Papadopoulos     230169.0
Versie Hillebrand      187693.0
Vicki Laflamme         478396.0
Violet Mclelland       123431.0
Wilburn Farren         157640.0
Zane Levy              430068.0
Name: close_value, dtype: fl

In [362]:
df[df["deal_stage"]=="Won"].groupby("sales_agent")["close_value"].mean()

sales_agent
Anna Snelling         1322.384615
Boris Faz             2590.405941
Cassey Cress          2763.736196
Cecily Lampkin        2147.663551
Corliss Cosme         2806.906667
Daniell Hammack       3194.991228
Darcel Schlecht       3304.338109
Donn Cantrell         2821.898734
Elease Gluck          3614.937500
Garret Kinder         2636.973333
Gladys Colclough      2560.548148
Hayden Neloms         2543.093458
James Ascencio        3063.207407
Jonathan Berthelot    1666.000000
Kami Bicknell         1818.712644
Kary Hendrixson       2173.674641
Lajuana Vencill       1532.535433
Markita Hansen        2529.169231
Marty Freudenburg     2386.844262
Maureen Marcano       2351.644295
Moses Frase           1606.062016
Niesha Huffines       1685.342857
Reed Clapper          2827.974194
Rosalina Dieter       3269.486111
Rosie Papadopoulos    2950.884615
Versie Hillebrand     1066.437500
Vicki Laflamme        2164.687783
Violet Mclelland      1011.729508
Wilburn Farren        2866.181818
Za

In [363]:
df.groupby("sales_agent")["opportunity_id"].count()

sales_agent
Anna Snelling         448
Boris Faz             210
Cassey Cress          346
Cecily Lampkin        203
Corliss Cosme         310
Daniell Hammack       259
Darcel Schlecht       747
Donn Cantrell         275
Elease Gluck          177
Garret Kinder         123
Gladys Colclough      317
Hayden Neloms         202
James Ascencio        267
Jonathan Berthelot    345
Kami Bicknell         362
Kary Hendrixson       438
Lajuana Vencill       311
Markita Hansen        306
Marty Freudenburg     281
Maureen Marcano       285
Moses Frase           260
Niesha Huffines       239
Reed Clapper          237
Rosalina Dieter       160
Rosie Papadopoulos    160
Versie Hillebrand     361
Vicki Laflamme        451
Violet Mclelland      261
Wilburn Farren        110
Zane Levy             349
Name: opportunity_id, dtype: int64

In [364]:
won_by_agent = df[df["deal_stage"]== "Won"].groupby("sales_agent")["opportunity_id"].count()
lost_by_agent = df[df["deal_stage"]== "Lost"].groupby("sales_agent")["opportunity_id"].count()

print(lost_by_agent)

sales_agent
Anna Snelling         128
Boris Faz              52
Cassey Cress           98
Cecily Lampkin         53
Corliss Cosme          79
Daniell Hammack        73
Darcel Schlecht       204
Donn Cantrell         117
Elease Gluck           46
Garret Kinder          48
Gladys Colclough       97
Hayden Neloms          45
James Ascencio         71
Jonathan Berthelot     93
Kami Bicknell          98
Kary Hendrixson       126
Lajuana Vencill       104
Markita Hansen         97
Marty Freudenburg      72
Maureen Marcano        64
Moses Frase            66
Niesha Huffines        70
Reed Clapper           82
Rosalina Dieter        38
Rosie Papadopoulos     43
Versie Hillebrand      88
Vicki Laflamme        126
Violet Mclelland       71
Wilburn Farren         24
Zane Levy             100
Name: opportunity_id, dtype: int64


In [365]:
closed_by_agent =  won_by_agent + lost_by_agent
print(closed_by_agent)

sales_agent
Anna Snelling         336
Boris Faz             153
Cassey Cress          261
Cecily Lampkin        160
Corliss Cosme         229
Daniell Hammack       187
Darcel Schlecht       553
Donn Cantrell         275
Elease Gluck          126
Garret Kinder         123
Gladys Colclough      232
Hayden Neloms         152
James Ascencio        206
Jonathan Berthelot    264
Kami Bicknell         272
Kary Hendrixson       335
Lajuana Vencill       231
Markita Hansen        227
Marty Freudenburg     194
Maureen Marcano       213
Moses Frase           195
Niesha Huffines       175
Reed Clapper          237
Rosalina Dieter       110
Rosie Papadopoulos    121
Versie Hillebrand     264
Vicki Laflamme        347
Violet Mclelland      193
Wilburn Farren         79
Zane Levy             261
Name: opportunity_id, dtype: int64


In [366]:
win_rate_by_agent = won_by_agent/(closed_by_agent)*100
print(win_rate_by_agent)

sales_agent
Anna Snelling         61.904762
Boris Faz             66.013072
Cassey Cress          62.452107
Cecily Lampkin        66.875000
Corliss Cosme         65.502183
Daniell Hammack       60.962567
Darcel Schlecht       63.110307
Donn Cantrell         57.454545
Elease Gluck          63.492063
Garret Kinder         60.975610
Gladys Colclough      58.189655
Hayden Neloms         70.394737
James Ascencio        65.533981
Jonathan Berthelot    64.772727
Kami Bicknell         63.970588
Kary Hendrixson       62.388060
Lajuana Vencill       54.978355
Markita Hansen        57.268722
Marty Freudenburg     62.886598
Maureen Marcano       69.953052
Moses Frase           66.153846
Niesha Huffines       60.000000
Reed Clapper          65.400844
Rosalina Dieter       65.454545
Rosie Papadopoulos    64.462810
Versie Hillebrand     66.666667
Vicki Laflamme        63.688761
Violet Mclelland      63.212435
Wilburn Farren        69.620253
Zane Levy             61.685824
Name: opportunity_id, dtype:

## 7. Account Performance

Account-level analysis is used to identify the customers contributing the most to Won opportunities and recorded Won value.

The analysis also considers average Won deal value and the concentration of Won value among the company's largest accounts.


In [372]:
df[df["deal_stage"]=="Won"].groupby("account")["opportunity_id"].count()

account
Acme Corporation    34
Betasoloin          34
Betatech            53
Bioholding          50
Bioplex             31
                    ..
Zencorporation      33
Zoomit              33
Zotware             47
Zumgoity            36
dambase             54
Name: opportunity_id, Length: 85, dtype: int64

In [373]:
won_value_by_account = df[df["deal_stage"]=="Won"].groupby("account")["close_value"].sum()

In [374]:
won_value_by_account.sort_values(ascending=False).head(5)

account
Kan-code    341455.0
Konex       269245.0
Condax      206410.0
Cheers      198020.0
Hottechi    194957.0
Name: close_value, dtype: float64

In [375]:
won_deals_by_account = df[df["deal_stage"]=="Won"].groupby("account")["opportunity_id"].count()
won_deals_by_account.sort_values(ascending=False).head(5)

account
Kan-code    115
Hottechi    111
Konex       108
Condax      105
Rangreen     75
Name: opportunity_id, dtype: int64

In [376]:
average_won_value_by_account = df[df["deal_stage"]=="Won"].groupby("account")["close_value"].mean()
average_won_value_by_account.sort_values(ascending= False).head(5)

account
Xx-holding                   3528.270833
Cheers                       3474.035088
Vehement Capital Partners    3280.382353
Ontomedia                    3238.176471
Umbrella Corporation         3181.270833
Name: close_value, dtype: float64

In [377]:
average_won_value_by_account["Kan-code"]

np.float64(2969.1739130434785)

In [378]:
top_5_value_account = won_value_by_account.sort_values(ascending=False).head(5)

In [379]:
top_5_value_account.sum()

np.float64(1210087.0)

In [380]:
won_value_by_account.sum()

np.float64(10005534.0)

In [381]:
top_5_value_account.sum()/(won_value_by_account.sum())*100

np.float64(12.094177082402599)

## Clarifying the averages on sales values 

In [384]:
sales_agent_average = df[df["deal_stage"]=="Won"].groupby("sales_agent")["close_value"].mean()

In [385]:
sales_agent_average.sort_values(ascending=False).head(5)

sales_agent
Elease Gluck       3614.937500
Darcel Schlecht    3304.338109
Rosalina Dieter    3269.486111
Daniell Hammack    3194.991228
James Ascencio     3063.207407
Name: close_value, dtype: float64

In [386]:
sales_agent_sum_money = df[df["deal_stage"]=="Won"].groupby("sales_agent")["close_value"].sum()

In [387]:
sales_agent_sum_money.sort_values(ascending=False).head(5)

sales_agent
Darcel Schlecht    1153214.0
Vicki Laflamme      478396.0
Kary Hendrixson     454298.0
Cassey Cress        450489.0
Donn Cantrell       445860.0
Name: close_value, dtype: float64

## Exploring the other datasets 


In [388]:
df_account =pd.read_csv("../data/accounts.csv")
df_account.head()
df_account.info()

<class 'pandas.DataFrame'>
RangeIndex: 85 entries, 0 to 84
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   account           85 non-null     str    
 1   sector            85 non-null     str    
 2   year_established  85 non-null     int64  
 3   revenue           85 non-null     float64
 4   employees         85 non-null     int64  
 5   office_location   85 non-null     str    
 6   subsidiary_of     15 non-null     str    
dtypes: float64(1), int64(2), str(4)
memory usage: 7.4 KB


In [389]:
df_product =pd.read_csv("../data/products.csv")
df_product.head()
df_product.info()

<class 'pandas.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   product      7 non-null      str  
 1   series       7 non-null      str  
 2   sales_price  7 non-null      int64
dtypes: int64(1), str(2)
memory usage: 389.0 bytes


In [390]:
df_sales_team =pd.read_csv("../data/sales_teams.csv")
df_sales_team.head()
df_sales_team.info()

<class 'pandas.DataFrame'>
RangeIndex: 35 entries, 0 to 34
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   sales_agent      35 non-null     str  
 1   manager          35 non-null     str  
 2   regional_office  35 non-null     str  
dtypes: str(3)
memory usage: 2.0 KB


In [391]:
df_dictionary =pd.read_csv("../data/data_dictionary.csv")
df_dictionary.head()
df_dictionary.info()

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Table        21 non-null     str  
 1   Field        21 non-null     str  
 2   Description  21 non-null     str  
dtypes: str(3)
memory usage: 1.5 KB


## 8. Regional & Manager Performance

The sales pipeline is combined with the sales team information to investigate performance at regional and manager level.

This allows the analysis to compare Won deal volume, total Won value and average Won deal value across regional offices and managers.


In [392]:
merged_df_sales_team = df.merge(df_sales_team, on="sales_agent")

In [393]:
merged_df_sales_team[df["deal_stage"]=="Won"].groupby("regional_office")["opportunity_id"].count()

regional_office
Central    1629
East       1171
West       1438
Name: opportunity_id, dtype: int64

In [394]:
merged_df_sales_team[df["deal_stage"]=="Won"].groupby("regional_office")["close_value"].sum()

regional_office
Central    3346293.0
East       3090594.0
West       3568647.0
Name: close_value, dtype: float64

In [395]:
merged_df_sales_team[df["deal_stage"]=="Won"].groupby("regional_office")["close_value"].mean()

regional_office
Central    2054.200737
East       2639.277541
West       2481.673853
Name: close_value, dtype: float64

In [396]:
merged_df_sales_team[df["deal_stage"]=="Won"].groupby("manager")["opportunity_id"].count()

manager
Cara Losch          480
Celia Rouche        610
Dustin Brinkmann    747
Melvin Marxen       882
Rocco Neubert       691
Summer Sewald       828
Name: opportunity_id, dtype: int64

In [397]:
merged_df_sales_team[df["deal_stage"]=="Won"].groupby("manager")["close_value"].sum()

manager
Cara Losch          1130049.0
Celia Rouche        1603897.0
Dustin Brinkmann    1094363.0
Melvin Marxen       2251930.0
Rocco Neubert       1960545.0
Summer Sewald       1964750.0
Name: close_value, dtype: float64

In [398]:
merged_df_sales_team[df["deal_stage"]=="Won"].groupby("manager")["close_value"].mean()

manager
Cara Losch          2354.268750
Celia Rouche        2629.339344
Dustin Brinkmann    1465.010710
Melvin Marxen       2553.208617
Rocco Neubert       2837.257598
Summer Sewald       2372.886473
Name: close_value, dtype: float64

## 9. Customer Sector Performance

The sales pipeline is combined with the account information to investigate performance across different customer sectors.

The analysis compares Won deal volume, total Won value and average Won deal value to identify which sectors contribute most to the company's sales performance.


In [399]:
df_merged_account_sales_team = merged_df_sales_team.merge(df_account, on="account")

In [400]:
df_merged_account_sales_team[df["deal_stage"]=="Won"].groupby("sector")["opportunity_id"].count()

C:\Users\josha\AppData\Local\Temp\ipykernel_2804\3229499868.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_merged_account_sales_team[df["deal_stage"]=="Won"].groupby("sector")["opportunity_id"].count()


sector
employment            164
entertainment         221
finance               336
marketing             348
medical               558
retail                694
services              189
software              373
technolgy             595
telecommunications    254
Name: opportunity_id, dtype: int64

In [401]:
df_merged_account_sales_team[df["deal_stage"]=="Won"].groupby("sector")["close_value"].sum()

C:\Users\josha\AppData\Local\Temp\ipykernel_2804\3960799730.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_merged_account_sales_team[df["deal_stage"]=="Won"].groupby("sector")["close_value"].sum()


sector
employment            209635.0
entertainment         340915.0
finance               445550.0
marketing             433875.0
medical               698749.0
retail                947251.0
services              284826.0
software              524559.0
technolgy             744937.0
telecommunications    301311.0
Name: close_value, dtype: float64

In [402]:
df_merged_account_sales_team[df["deal_stage"]=="Won"].groupby("sector")["close_value"].mean()

C:\Users\josha\AppData\Local\Temp\ipykernel_2804\3901886442.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_merged_account_sales_team[df["deal_stage"]=="Won"].groupby("sector")["close_value"].mean()


sector
employment            1388.311258
entertainment         1748.282051
finance               1480.232558
marketing             1355.859375
medical               1380.926877
retail                1473.174184
services              1695.392857
software              1524.880814
technolgy             1395.013109
telecommunications    1315.768559
Name: close_value, dtype: float64

## 10. Initial Findings

The analysis provides an initial view of the company's sales pipeline and the factors associated with sales performance.

* The dataset contains 8,800 sales opportunities. Of these, 4,238 are Won, 2,473 are Lost and 2,089 remain open in the Engaging or Prospecting stages.

* The overall closed win rate is 63.15%, meaning that approximately 63% of opportunities that reached a final outcome were Won.

* The company has 2,089 open opportunities, representing 23.74% of the total pipeline. Most of these open opportunities are in the Engaging stage, which accounts for approximately 76% of the open pipeline.

* The missing `close_date` and `close_value` information is concentrated entirely within open opportunities. This suggests that these missing values are related to the current stage of the opportunities rather than necessarily representing data-quality errors.

* The recorded Won value is £10.01 million, with an average Won deal value of approximately £2,360.91.

* Product performance varies considerably. GTXPro generated the highest recorded Won value at approximately £3.51 million, followed by GTX Plus Pro at £2.63 million and MG Advanced at £2.22 million.

* GTK 500 has the highest average Won deal value at approximately £26,707, but only 15 opportunities were Won. This small number of deals means the result should be treated cautiously.

* MG Special has one of the highest win rates at approximately 64.84%, but contributes only £43,768 of recorded Won value. This shows why win rate should not be considered on its own when evaluating product performance.

* Sales-agent performance also varies. Darcel Schlecht recorded the highest number of Won opportunities at 349 and the highest total Won value at approximately £1.15 million. However, other agents achieved higher average Won deal values and win rates, showing that no single measure fully represents sales performance.

* At account level, Kan-code generated the highest recorded Won value at approximately £341,455. The five highest-value accounts together contributed approximately 12.1% of total recorded Won value, suggesting that Won value is distributed across a relatively broad customer base rather than being heavily concentrated in a small number of accounts.

* Regional performance also differs. The West region generated the highest total Won value at approximately £3.57 million, while Central recorded the highest number of Won opportunities at 1,629. East had the highest average Won deal value at approximately £2,639.

* At manager level, Melvin Marxen's team recorded the highest number of Won opportunities and the highest total Won value. Rocco Neubert's team had the highest average Won deal value.

* Customer sector performance shows that retail generated the highest number of Won opportunities and the highest recorded Won value. Differences between sectors suggest that customer composition may be an important factor when investigating sales performance further.

Overall, the analysis shows that sales performance is influenced by a combination of deal volume, deal value, product mix, sales-agent performance, customer accounts, regions and customer sectors. Further analysis is required to understand why these differences occur and whether they are driven by underlying differences in products, customers or sales teams.


## 11. Questions for Further Analysis

The initial analysis provides an overview of pipeline and sales performance, but several questions require further investigation.

* Which sales agents consistently perform strongly when both win rate and deal value are considered?
* Are certain products particularly successful within specific customer sectors?
* Which accounts have the greatest potential within the open pipeline?
* Are regional differences driven by sales volume, deal value, product mix or customer composition?
* What factors might explain the differences in performance between managers?
* How does product mix vary across sales agents, regions and customer sectors?
* Are the highest-value products also the products with the strongest conversion performance?
